# Colab 微调 v4 — 逐步保存版
每格跑完结果都存盘，断了重跑对应格即可。

In [ ]:
# ============================================================
# 第 1 格：安装依赖 + 挂载 Drive + 上传数据
# 断了重跑：再次运行本格即可
# ============================================================
!pip install -q unsloth datasets peft accelerate trl

from google.colab import drive
drive.mount("/content/drive")
import os, json

SAVE_DIR = "/content/drive/MyDrive/lora_v4"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"[OK] Drive 挂载成功，存档目录: {SAVE_DIR}")

# 上传数据
from google.colab import files
print("\n>>> 请上传 game_assistant_v3.jsonl")
uploaded = files.upload()

# 找到数据文件
for fn in list(uploaded.keys()):
    if fn.endswith(".jsonl"):
        DATA_FILE = fn
        break

# 也拷贝到 Drive 保底
!cp {DATA_FILE} {SAVE_DIR}/data.jsonl 2>/dev/null
print(f"[OK] 数据文件: {DATA_FILE}（已备份到 Drive）")

In [ ]:
# ============================================================
# 第 2 格：加载模型 + 准备数据
# 断了重跑：再次运行本格即可（模型在 Colab 缓存里，很快）
# ============================================================
from unsloth import FastLanguageModel
import torch, json, os

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[OK] 模型就绪: {trainable/1e6:.1f}M 可训练参数")

# 加载数据
DATA_FILE = "game_assistant_v3.jsonl"
if not os.path.exists(DATA_FILE):
    DATA_FILE = f"{SAVE_DIR}/data.jsonl"  # 从 Drive 恢复

data = []
with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line.strip()))
print(f"[OK] 数据: {len(data)} 条")

# 格式化
from datasets import Dataset
def fmt(ex):
    t = ""
    for m in ex["messages"]:
        r, c = m["role"], m.get("content","")
        if r == "user": t += f"<|im_start|>user\n{c}<|im_end|>\n"
        elif r == "assistant":
            if m.get("tool_calls"):
                t += f"<|im_start|>assistant\n{json.dumps(m['tool_calls'],ensure_ascii=False)}<|im_end|>\n"
            else:
                t += f"<|im_start|>assistant\n{c}<|im_end|>\n"
        elif r == "tool": t += f"<|im_start|>tool\n{c}<|im_end|>\n"
    return {"text": t}
dataset = Dataset.from_list(data).map(fmt)
print("[OK] 数据格式化完成")

In [ ]:
# ============================================================
# 第 3 格：训练（每 10 步存一次 Drive 检查点）
# 断了重跑：本格自动从最新 checkpoint 续训
# ============================================================
from trl import SFTTrainer, SFTConfig

CKPT_DIR = f"{SAVE_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

MERGED_DIR = f"{SAVE_DIR}/merged_model"
ALREADY_MERGED = os.path.exists(f"{MERGED_DIR}/model-00001-of-00004.safetensors")

if ALREADY_MERGED:
    print("[SKIP] 训练已完成，合并模型已存在。跳到第 4 格")
else:
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=2048,
            per_device_train_batch_size=2, gradient_accumulation_steps=8,
            num_train_epochs=3, learning_rate=1e-4, warmup_steps=10,
            fp16=True, logging_steps=1,
            save_steps=10,               # 每 10 步存 Drive
            save_total_limit=2,          # 只保留最近 2 个检查点
            output_dir=CKPT_DIR,
            resume_from_checkpoint=True,  # 断了自动续训！
            report_to="none",
        ),
    )
    print(">>> 开始训练（检查点每 10 步保存到 Drive）...")
    trainer.train()
    print(">>> 训练完成！")

In [ ]:
# ============================================================
# 第 4 格：合并 LoRA → 完整模型（存 Drive）
# 断了重跑：检测到合并模型自动跳过
# ============================================================
ALREADY_MERGED = os.path.exists(f"{MERGED_DIR}/model-00001-of-00004.safetensors")

if ALREADY_MERGED:
    print("[SKIP] 合并模型已存在")
else:
    print(">>> 合并 LoRA 权重到完整模型...")
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    print(f"[OK] 合并模型 -> {MERGED_DIR}")

In [ ]:
# ============================================================
# 第 5 格：转换 GGUF（存 Drive）
# 断了重跑：检测到 GGUF 文件自动跳过
# ============================================================
GGUF_PATH = f"{SAVE_DIR}/game_assistant_7b.gguf"

if os.path.exists(GGUF_PATH):
    print(f"[SKIP] GGUF 已存在: {GGUF_PATH}")
else:
    print(">>> 克隆 llama.cpp...")
    !git clone -q https://github.com/ggerganov/llama.cpp.git 2>/dev/null
    !pip install -q -r llama.cpp/requirements.txt 2>/dev/null

    print(">>> 转换 GGUF（q8_0 量化）...")
    !python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype q8_0 --outfile {GGUF_PATH}
    
    if os.path.exists(GGUF_PATH):
        size_gb = os.path.getsize(GGUF_PATH) / 1e9
        print(f"[OK] GGUF 已保存: {GGUF_PATH} ({size_gb:.2f} GB)")
    else:
        print("[ERR] GGUF 转换失败，检查错误信息")

In [ ]:
# ============================================================
# 第 6 格：下载 GGUF 到本地电脑
# ============================================================
import os
GGUF_PATH = f"{SAVE_DIR}/game_assistant_7b.gguf"

if os.path.exists(GGUF_PATH):
    from google.colab import files
    files.download(GGUF_PATH)
else:
    print(f"[ERR] GGUF 文件不存在: {GGUF_PATH}")
    print("请先运行第 5 格")

## 导入 Ollama
下载 GGUF 后，本地终端：
```powershell
cd 下载目录
ollama create game-assistant-v2 -f Modelfile
```
Modelfile:
```
FROM ./game_assistant_7b.gguf
TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>"""
PARAMETER temperature 0.8
PARAMETER top_p 0.9
```

然后把 profile.json 的 model 改为 `game-assistant-v2`。